# Milestone 4 — Reproducible GPU Training (Colab)

**Experiments:** E1 Pakistani/Xception · E2 Pakistani/ConvNeXt-Tiny · E3 Stanford/Xception · E4 Stanford/ConvNeXt-Tiny.

Use only the frozen manifests. **Do not run test evaluation during training or smoke tests.**

**Prerequisites before opening this notebook:**
1. Copy the supplied `train_vehicle.py` into `src/` locally and apply the small YAML indentation patch.
2. Commit and push changes to `main`; all team members use the same commit.
3. From your repository root, create a ZIP that **contains `data/` as its first-level folder**: `zip -r data.zip data/`. Upload to Google Drive at `MyDrive/vehicle_project/data.zip`. Do not commit image datasets to GitHub.
4. Select **Runtime → Change runtime type → GPU** and connect.

No output in this notebook is a claimed experimental result until you actually run it.

## 1. Verify GPU

In [ ]:
import sys, subprocess, torch
print('Python:', sys.version)
print('Torch:', torch.__version__, 'CUDA build:', torch.version.cuda)
assert torch.cuda.is_available(), 'Use Runtime → Change runtime type → GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.run(['nvidia-smi'], check=True)

## 2. Mount Google Drive (ZIP and checkpoints are persisted here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/vehicle_project')
ZIP_PATH = DRIVE_DIR / 'data.zip'
OUTPUT_DIR = DRIVE_DIR / 'runs'
assert ZIP_PATH.is_file(), f'Upload data.zip first: {ZIP_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset ZIP:', ZIP_PATH)
print('Persistent outputs:', OUTPUT_DIR)

## 3. Clone fixed repository commit and install light dependencies

If the repository is private, clone using an authorized GitHub workflow. Do not put personal-access tokens into notebook cells.

In [ ]:
import subprocess, os
REPO = Path('/content/vehicle-make-model-recognition')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Batsy1857/vehicle-make-model-recognition.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True)
assert (REPO / 'src/train_vehicle.py').exists(), 'Commit the supplied script to GitHub before starting Colab'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'pyyaml', 'scikit-learn'], check=True)
import timm, torchvision, yaml, sklearn
print('Versions:', 'timm', timm.__version__, 'torchvision', torchvision.__version__, 'sklearn', sklearn.__version__)

## 4. Extract the identical image files into Colab local storage

The paths must match the manifest, e.g. `data/pakistan/Splited_dataset/train/...`. Extracting on local Colab storage is preferable to loading thousands of individual images from mounted Drive.

In [ ]:
import zipfile, shutil
with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()
    assert any(name.startswith('data/pakistan/') for name in names), 'ZIP needs data/pakistan/... paths'
    assert any(name.startswith('data/stanford_cars/') for name in names), 'ZIP needs data/stanford_cars/... paths'
    z.extractall(REPO)
print('Extracted image dataset into:', REPO / 'data')

## 5. Verify YAML, manifests, and available image files before training

In [ ]:
import sys
sys.path.insert(0, str(REPO / 'src'))
from vehicle_data import inspect_manifest
with (REPO / 'configs/experiments.yaml').open() as f:
    cfg = yaml.safe_load(f)
assert cfg['training']['batch_size'] == 16
for key in ['pakistan', 'stanford_cars']:
    info = inspect_manifest(REPO, key, check_files=True)
    assert info['manifest_sha256'] == cfg['datasets'][key]['manifest_sha256']
    for split in ['train', 'val', 'test']:
        assert info['counts'][split] == cfg['datasets'][key]['verified_split_counts'][split]
    print(key, 'classes', info['class_count'], 'counts', info['counts'], 'hash', info['manifest_sha256'])
print('PASS: frozen data integrity checks')

## 6. Short GPU smoke tests — **not** official experiments

Each smoke test runs only **two training batches and two validation batches per stage**, one epoch in head-only and one in partial fine-tuning. This checks GPU forward/backpropagation, optimizer, stage transitions, and checkpoint writing. The reported metrics are not suitable for the final comparison.

In [ ]:
SMOKE_EXPERIMENTS = ['E1', 'E4']  # covers both datasets and architectures
for exp in SMOKE_EXPERIMENTS:
    cmd = [sys.executable, str(REPO / 'src/train_vehicle.py'), '--experiment', exp,
           '--run-id', f'{exp}-smoke01', '--output-dir', str(OUTPUT_DIR), '--smoke']
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)
print('GPU smoke tests complete; inspect their history.csv and metadata.json')

## 7. Full training — run after examining smoke outputs

The same seed, frozen manifests, train/val transforms, batch sizes, and two-stage YAML schedule apply to every run. Drive output paths are persistent. If disconnected, rerun the interrupted experiment with `--resume` and the same `run-id`, provided the repo commit has not changed.

In [ ]:
# Set to exactly one experiment at a time to control GPU runtime use.
EXPERIMENT = 'E1'  # Change sequentially: E1, E2, E3, E4
RUN_ID = f'{EXPERIMENT}-run01'
cmd = [sys.executable, str(REPO / 'src/train_vehicle.py'), '--experiment', EXPERIMENT,
       '--run-id', RUN_ID, '--output-dir', str(OUTPUT_DIR)]
# To resume after disconnect, add: cmd.append('--resume')
print('Starting FULL training:', EXPERIMENT, RUN_ID)
subprocess.run(cmd, cwd=REPO, check=True)
print('Inspect:', OUTPUT_DIR / RUN_ID / 'history.csv')

## 8. Inspect validation metrics (without opening test data)

In [ ]:
import pandas as pd
history_file = OUTPUT_DIR / RUN_ID / 'history.csv'
assert history_file.exists()
history = pd.read_csv(history_file)
display(history[['stage','epoch','train_loss','val_loss','val_accuracy','val_macro_f1','train_seconds','peak_gpu_memory_mb']])
print('Validation best macro-F1:', history.val_macro_f1.max())
print('Training seconds (excluding validation):', history.train_seconds.sum())

## 8b. Plot actual learning curves for the report

Only execute this after a full run. The graph comes from saved results, not illustrative numbers.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for stage_name, stage_df in history.groupby('stage', sort=False):
    x = range(1, len(stage_df) + 1)
    axes[0].plot(x, stage_df['val_macro_f1'], marker='o', label=stage_name)
    axes[1].plot(x, stage_df['train_loss'], marker='o', label=stage_name)
axes[0].set(title=f'{EXPERIMENT}: validation macro-F1', xlabel='Epoch within stage', ylabel='Macro-F1')
axes[1].set(title=f'{EXPERIMENT}: training loss', xlabel='Epoch within stage', ylabel='Cross-entropy')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 9. Only after all full runs are complete and selected by validation: final test and inference benchmark

For each official run, test only the selected `best.pt`. The test evaluator refuses to use smoke runs and records `test_metrics.json`, per-class metrics, and confusion matrix. Run benchmarks on the **same GPU model** for all four experiments or label GPU comparisons incomparable. No test data is loaded by training mode.

In [ ]:
RUN_FINAL_TESTS = False  # Explicitly set True only AFTER all four final models are selected.
if RUN_FINAL_TESTS:
    for exp in ['E1', 'E2', 'E3', 'E4']:
        rid = f'{exp}-run01'
        for mode in ['test', 'benchmark']:
            cmd = [sys.executable, str(REPO / 'src/train_vehicle.py'), '--experiment', exp,
                   '--run-id', rid, '--output-dir', str(OUTPUT_DIR), '--mode', mode]
            subprocess.run(cmd, cwd=REPO, check=True)
else:
    print('Held-out test sets remain sealed. No final test evaluation requested.')

## 10. Export results and record in `EXPERIMENT_LOG.md`

Copy measured metrics **only from the files written by actual runs**, noting the device, Git SHA, manifest hashes, parameters, best epoch, and any interruptions or deviations. Do not commit checkpoints or datasets to GitHub. Commit training source/notebook and small CSV/JSON metrics if permitted by the group.

## 11. Generate a shared results table after final testing

This cell uses saved run artifacts only and fails if any run is missing. The CSV is the single source of truth for report and slides.

In [ ]:
import pandas as pd, json
EXPORT_RESULTS = False  # Switch True only after all four test_metrics.json and benchmark.json exist
if EXPORT_RESULTS:
    rows = []
    for exp in ['E1','E2','E3','E4']:
        d = OUTPUT_DIR / f'{exp}-run01'
        info = json.loads((d/'metadata.json').read_text())
        test = json.loads((d/'test_metrics.json').read_text())
        speed = json.loads((d/'benchmark.json').read_text())
        rows.append({
            'experiment': exp, 'dataset': info['experiment_definition']['dataset'],
            'model': info['experiment_definition']['model'], 'GPU': info['gpu'],
            'manifest_sha256': info['dataset_audit']['manifest_sha256'],
            'git_commit': info['git_commit'],
            'total_parameters': info['parameters_head_only']['total'],
            'head_trainable_parameters': info['parameters_head_only']['trainable'],
            'fine_tune_trainable_parameters': info['parameters_partial_fine_tune']['trainable'],
            'best_stage': info['selected_stage'], 'best_epoch': info['selected_epoch'],
            'best_val_macro_f1': info['best_validation_macro_f1'],
            'test_top1_accuracy': test['test_metrics']['accuracy'],
            'test_top5_accuracy': test['test_metrics']['top5_accuracy'],
            'test_macro_f1': test['test_metrics']['macro_f1'],
            'train_seconds_excl_validation': info['total_train_seconds_excluding_validation'],
            'peak_gpu_memory_mb': info['peak_gpu_memory_mb'],
            'batch1_median_ms_per_image': speed['batch1_median_ms_per_image'],
            'batch1_p95_ms_per_image': speed['batch1_p95_ms_per_image'],
            'batch16_median_ms_per_image': speed['batch16_median_ms_per_image'],
            'batch16_p95_ms_per_image': speed['batch16_p95_ms_per_image'],
        })
    out = REPO / 'results/metrics.csv'
    out.parent.mkdir(exist_ok=True)
    pd.DataFrame(rows).to_csv(out, index=False)
    shutil.copy2(out, DRIVE_DIR / 'metrics.csv')
    display(pd.DataFrame(rows))
    print('Exported:', out, 'and', DRIVE_DIR / 'metrics.csv')
else:
    print('Results export disabled until every official run is evaluated.')